# Lesson 04 - Tool Use Design Pattern

In this lesson you will learn the **Tool Use** design pattern for AI agents using the Microsoft Agent Framework (Python). We cover:

- Defining function tools with the `@tool` decorator and typed parameters
- Providing tool schemas so the model knows what each tool does
- Controlling tool execution with `approval_mode`
- Returning **structured output** via Pydantic models and `response_format`

The scenario is a **travel booking agent** that can look up destinations, check availability, and retrieve flight information.

## Setup

In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.ai.projects.models import PromptAgentDefinition, FunctionTool
from agents import Tool


In [ ]:
# Create the Azure AI Foundry provider
project_client = AIProjectClient(
    endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    credential=DefaultAzureCredential()
)

In [ ]:

def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]

destionations_tool = FunctionTool(
    name="get_destinations",
    description="Get available vacation destinations.",
    parameters={
        "type": "object",
        "properties": {},
        "required": [],
        "additionalProperties": False,
    },
    strict=True,
    )
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")

check_availability_tool = FunctionTool(
    name="check_availability",
    description="Check booking availability for a destination.",
    parameters={
        "type": "object",
        "properties": {
            "destination": {"type": "string", "description": "The destination to check"},
        },
        "required": ["destination"],
        "additionalProperties": False,
    },
    strict=True,
    )


def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

flight_info_tool = FunctionTool(
    name="get_flight_info",
    description="Get flight information between two cities.",
    parameters={
        "type": "object",
        "properties": {
            "origin": {"type": "string", "description": "Origin airport code"},
            "destination": {"type": "string", "description": "Destination airport code"},
        },
        "required": ["origin", "destination"],
        "additionalProperties": False,
    },
    strict=True,
    )

tools: list[Tool] = [destionations_tool, check_availability_tool, flight_info_tool]

## Creating an Agent with Multiple Tools

Pass all three tools to the client so the model can invoke whichever ones it needs to answer the user's question.

In [ ]:
agent =  project_client.agents.create_version(
    agent_name="TravelToolAgent",
    definition=PromptAgentDefinition(
        instructions=("You are a travel agent. Use the available tools to answer questions"
                      "use get_destinations to get a list of available destinations,"
                      "use  check_availability to check if there is availability for a destination"
                      "and get_flight_info to get flight information between two cities."),
        model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
        tools=tools
    )
)

openai = project_client.get_openai_client()

response = openai.responses.create(
    input="Checking availability for Tokyo and flight info from London to Tokyo.",
    extra_body={
        "agent_reference": {
            "name": agent.name,
            "type": "agent_reference",
        }
    },
)

# Handle any requested function calls
for item in response.output:
    if item.type == "function_call":
        if item.name == "get_destinations":
            args = json.loads(item.arguments)
            result = get_destinations()
            print(f"Tool call: get_destinations() -> {result}")
        elif item.name == "check_availability":
            args = json.loads(item.arguments)
            destination = args["destination"]
            result = check_availability(destination)

            print(f"Tool call: check_availability(destination={destination}) -> {result}")
        elif item.name == "get_flight_info":
            args = json.loads(item.arguments)
            origin = args["origin"]
            destination = args["destination"]
            result = get_flight_info(origin, destination)
            print(f"Tool call: get_flight_info(origin={origin}, destination={destination}) -> {result}")
 



## Summary

In this lesson you learned how to:

1. **Define tools** using the `FunctionTool`  with typed parameters and docstrings that serve as the tool schema.
2. **Compose multiple tools** so the agent can call them in sequence to answer complex queries.

These patterns form the foundation for building reliable, production-ready agents that can interact with external systems safely.